In [ ]:
import re
from collections import defaultdict 

In [ ]:
# Count access per hour

with open('daten/sl-access.log', 'r') as logfile:
    hour_stats = defaultdict(int)
    for line in logfile:
        # [27/Apr/2026:18:37:03 +0200] "GET /...
        # m = re.match(r'.(..)..........(..)', line)
        # m = re.match(r'.(\d\d).{10}(\d\d)', line)
        m = re.match(r'.(\d\d)[^:]+:(\d\d)', line)
        day_hour = m.group(1) + "-" + m.group(2)
        hour_stats[day_hour] += 1
        
hour_stats

In [ ]:
# Alternative Variante mit re.split()
with open('daten/sl-access.log', 'r') as logfile:
    hour_stats = defaultdict(int)
    for line in logfile:
        # [27/Apr/2026:18:37:03 +0200] "GET /...
        parts = re.split(r'[:/\[]', line, maxsplit=5)
        day_hour = parts[1] + "-" + parts[4]
        hour_stats[day_hour] += 1
        
hour_stats

In [ ]:
# [24/Apr/2026:00:04:12 +0200] "GET /?edit=TheBreakthroughToShodan&section=2 HTTP/1.1" 403 4244 "-" "Mozilla/5.0 (Windows NT 10.0; Win64;

with open('daten/sl-access.log', 'r') as logfile:
    action_stats = defaultdict(lambda: [0,0,0])
    for line in logfile:
        # m = re.search(r' /\?([a-z]+)=[^"]+" (\d+) (\d+)', line, re.IGNORECASE)
        m = re.search(r"""
            [ ]/\?     # Anfang der URL
            ([a-z]+)   # Aktion
            =[^"]+"    # Rest der URL von _=_ bis _"_
            [ ](\d+)   # HTTP-Status
            [ ](\d+)   # Anzahl Bytes
            """, line, re.VERBOSE)
        if m is None:
            continue
        action_stats[m.group(1)][0] += 1
        if int(m.group(2)) >= 400: # Fehler zählen
            action_stats[m.group(1)][1] += 1
        action_stats[m.group(1)][2] += int(m.group(3))
        
action_stats